In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q dagshub mlflow
import mlflow
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
dagshub_username = user_secrets.get_secret("DAGSHUB_USERNAME")

os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_username
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token

mlflow.set_tracking_uri(f"https://dagshub.com/{dagshub_username}/ML_assignment2.mlflow")
mlflow.end_run()

print(f"Successfully connected to {dagshub_username}'s MLflow server!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 60.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import gc

test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

for col in test_transaction.select_dtypes('float64').columns:
    test_transaction[col] = test_transaction[col].astype('float32')
for col in test_identity.select_dtypes('float64').columns:
    test_identity[col] = test_identity[col].astype('float32')

test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

del test_transaction, test_identity
gc.collect()

print(f"Test Dataset Shape: {test.shape}")
print(f"Memory Usage: {test.memory_usage().sum() / 1024**2:.2f} MB")

Test Dataset Shape: (506691, 433)
Memory Usage: 902.65 MB


In [4]:
model_name = "XGB_tuned"
model_version = 1

best_model = mlflow.sklearn.load_model(
    model_uri=f"models:/{model_name}/{model_version}"
)

print(f"Successfully loaded {model_name} version {model_version}")

Successfully loaded XGB_tuned version 1


In [10]:
test['Transaction_hour'] = (test['TransactionDT'] // 3600) % 24
test['Transaction_day'] = (test['TransactionDT'] // (3600 * 24)) % 7
test['card1_amt_mean'] = test.groupby('card1')['TransactionAmt'].transform('mean')
test['amt_to_mean_card1'] = test['TransactionAmt'] / test['card1_amt_mean']
test.drop(columns=['card1_amt_mean'], inplace=True)

transaction_ids = test['TransactionID']
drop_cols = ['TransactionID', 'TransactionDT']
X_test = test.drop(columns=drop_cols)

train_cols = best_model.named_steps['preprocessor'].feature_names_in_
missing_cols = set(train_cols) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = np.nan

X_test = X_test[train_cols]

cat_cols = best_model.named_steps['preprocessor'].transformers_[0][2]
for col in cat_cols:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype('object')

print(f"Test feature count: {X_test.shape[1]}")

Test feature count: 423


In [11]:
mlflow.end_run()
with mlflow.start_run(run_name="XGB_inference"):

    test_preds = best_model.predict_proba(X_test)[:, 1]

    submission = pd.DataFrame({
        'TransactionID': transaction_ids,
        'isFraud': test_preds
    })

    submission.to_csv('submission.csv', index=False)

    mlflow.log_metric("num_predictions", len(submission))
    mlflow.log_metric("mean_fraud_probability", test_preds.mean())
    mlflow.log_metric("fraud_rate_predicted", (test_preds > 0.5).mean())
    mlflow.log_artifact("submission.csv")

    print(f"Submission shape: {submission.shape}")
    print(f"Mean fraud probability: {test_preds.mean():.4f}")
    print(f"Predicted fraud rate: {(test_preds > 0.5).mean():.4f}")
    print("submission.csv saved!")

Submission shape: (506691, 2)
Mean fraud probability: 0.0232
Predicted fraud rate: 0.0151
submission.csv saved!
🏃 View run XGB_inference at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/3a4977077b1a4f9db0c6b9fbf593c388
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


In [12]:
sample_submission = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv')

print(f"Expected shape: {sample_submission.shape}")
print(f"Our shape: {submission.shape}")
print(f"\nExpected columns: {sample_submission.columns.tolist()}")
print(f"Our columns: {submission.columns.tolist()}")
print(f"\nSample of our submission:")
print(submission.head())

Expected shape: (506691, 2)
Our shape: (506691, 2)

Expected columns: ['TransactionID', 'isFraud']
Our columns: ['TransactionID', 'isFraud']

Sample of our submission:
   TransactionID   isFraud
0        3663549  0.003266
1        3663550  0.000958
2        3663551  0.002049
3        3663552  0.000930
4        3663553  0.000679
